# NASA C-MAPSS (FD001) — Remaining Useful Life Prediction
## An Honest, Practically-Framed Approach

This notebook builds a predictive maintenance model for turbofan engines using the
NASA C-MAPSS FD001 dataset. The goal is not a "perfect" RUL predictor — it's an
honest model whose value is expressed in business terms (expected profit vs a fixed
maintenance schedule), and whose limitations (especially early-stage reliability)
are measured and reported transparently rather than hidden.

**What this notebook covers:**
1. Clean data loading and unit-based (leakage-free) train/validation split
2. A regression baseline, evaluated on NASA's official test set, with RUL capping
3. A stage-matched retraining variant (trades average accuracy for worst-case robustness)
4. A binary "needs maintenance soon" classifier, translated into expected dollar value
   vs reactive and blanket-preventive strategies
5. A rigorous early-warning test: does the model actually work early in an engine's
   life, or only once failure is already close? (Spoiler: the latter — reported honestly.)

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                              accuracy_score, precision_score, recall_score, f1_score)

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw', 'cmapss')

col_names = (['unit_number', 'time_in_cycles', 'setting_1', 'setting_2', 'setting_3']
             + [f's_{i}' for i in range(1, 22)])

train = pd.read_csv(os.path.join(DATA_DIR, 'train_FD001.txt'), sep=r'\s+', header=None, names=col_names)
test = pd.read_csv(os.path.join(DATA_DIR, 'test_FD001.txt'), sep=r'\s+', header=None, names=col_names)
y_test_official = pd.read_csv(os.path.join(DATA_DIR, 'RUL_FD001.txt'), sep=r'\s+', header=None, names=['RUL'])

print("Train shape:", train.shape, "| engines:", train['unit_number'].nunique())
print("Test shape:", test.shape, "| engines:", test['unit_number'].nunique())

## 2. Cleanup: Drop Constant Sensors, Add RUL Label

Seven of the 21 sensors are constant (zero variance) across all engines — they carry
no information and are dropped. RUL is computed as `max_cycle - time_in_cycles` for
the training set (which contains full run-to-failure trajectories).

In [ ]:
constant_sensors = ['s_1', 's_5', 's_6', 's_10', 's_16', 's_18', 's_19']
active_sensors = [c for c in col_names if c.startswith('s_') and c not in constant_sensors]

def add_rul(df):
    df = df.copy()
    max_cycle = df.groupby('unit_number')['time_in_cycles'].transform('max')
    df['RUL'] = max_cycle - df['time_in_cycles']
    return df

train = add_rul(train)
print("Active sensors:", active_sensors)

## 3. Feature Engineering: Rolling Mean

A 10-cycle rolling mean per sensor (computed within each engine, using only past data)
smooths sensor noise. This is added alongside the raw sensor values, not as a replacement.

In [ ]:
def add_rolling_features(df, sensors, window=10):
    df = df.sort_values(['unit_number', 'time_in_cycles']).copy()
    for s in sensors:
        df[f'{s}_rm'] = df.groupby('unit_number')[s].transform(
            lambda x: x.rolling(window, min_periods=1).mean()
        )
    return df

train = add_rolling_features(train, active_sensors)
test = add_rolling_features(test, active_sensors)

feature_cols = active_sensors + [f'{s}_rm' for s in active_sensors]
print(f"Total features: {len(feature_cols)}")

## 4. Unit-Based Train/Validation Split, and a Leakage Audit

Splitting by row (instead of by engine) is a common and serious mistake in this kind
of problem: consecutive cycles from the same engine are highly correlated, so a random
row-level split leaks information between train and test. We split strictly by
`unit_number` and confirm there is zero overlap.

In [ ]:
all_units = train['unit_number'].unique()
train_units, val_units = train_test_split(all_units, test_size=0.3, random_state=42)

train_split = train[train['unit_number'].isin(train_units)]
val_split = train[train['unit_number'].isin(val_units)]

train_ids = set(train_split['unit_number'].unique())
val_ids = set(val_split['unit_number'].unique())

print(f"Train units: {len(train_ids)} | Val units: {len(val_ids)}")
print("Train/Val overlap (must be 0):", len(train_ids & val_ids))
print("Official test engines come from a separate NASA file (test_FD001.txt) — no overlap by construction.")

## 5. Asymmetric Scoring Function (NASA PHM08 Competition Score)

Standard RMSE/MAE treat early and late prediction errors symmetrically. In reality,
predicting *more* remaining life than an engine actually has (a late/dangerous
prediction) is far costlier than the reverse. This exponential score penalizes late
predictions more heavily, matching real maintenance risk.

In [ ]:
def phm_score(y_true, y_pred, a1=13, a2=10):
    d = y_pred - y_true
    s = np.where(d < 0, np.exp(-d / a1) - 1, np.exp(d / a2) - 1)
    return np.sum(s)

def evaluate_full(y_true, y_pred, label=''):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    score = phm_score(y_true, y_pred)
    print(f"[{label}] MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.3f}  PHM-Score={score:.1f}")
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'score': score}

## 6. Baseline Regressor, Evaluated on the Official Test Set

Important: we evaluate on NASA's official `test_FD001.txt` + `RUL_FD001.txt`, whose
trajectories are truncated at genuinely varying points with real RUL diversity — not
on the training set's own held-out engines (whose final row is always RUL=0 by
definition, which would make evaluation trivial and misleading).

In [ ]:
rf_baseline = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, min_samples_leaf=5)
rf_baseline.fit(train_split[feature_cols], train_split['RUL'])

test_last = test.sort_values('time_in_cycles').groupby('unit_number').last().reset_index()
y_true = y_test_official['RUL'].values
y_pred = rf_baseline.predict(test_last[feature_cols])

baseline_metrics = evaluate_full(y_true, y_pred, label='Baseline')

## 7. Predicted vs Actual — Visual Check

In [ ]:
def plot_pred_vs_actual(y_true, y_pred, title=''):
    order = np.argsort(y_true)
    plt.figure(figsize=(11, 4))
    plt.plot(np.array(y_true)[order], label='Actual RUL', color='steelblue')
    plt.plot(np.array(y_pred)[order], label='Predicted RUL', color='tomato', alpha=0.7)
    plt.xlabel('Engines (sorted by actual RUL)')
    plt.ylabel('RUL (cycles)')
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_pred_vs_actual(y_true, y_pred, title='Baseline: Predicted vs Actual RUL')

## 8. RUL Capping

The uncapped PHM-score is dominated by rare extrapolation failures (a single engine
predicted at, say, 260 cycles when it actually had 113 left can contribute millions
to the exponential score). We report both uncapped and capped (RUL_CAP=130, a common
literature choice) metrics — capping is applied transparently, not to hide weak
performance.

In [ ]:
RUL_CAP = 130
y_true_capped = np.minimum(y_true, RUL_CAP)
y_pred_capped = np.minimum(y_pred, RUL_CAP)

_ = evaluate_full(y_true, y_pred, label='Uncapped')
_ = evaluate_full(y_true_capped, y_pred_capped, label='Capped @130')
plot_pred_vs_actual(y_true_capped, y_pred_capped, title='Baseline (Capped): Predicted vs Actual RUL')

## 9. Stage-Matched Retraining (a Robustness/Accuracy Trade-off)

Instead of training once on the full pool, we retrain per test engine using only
training rows whose `time_in_cycles` roughly matches the test engine's current stage
of life. Finding: this substantially reduces worst-case failures (uncapped PHM-Score
drops ~8x) at the cost of slightly worse average accuracy (capped R2 and MAE both
degrade a little). We keep it as a documented alternative, not a strict replacement
for the baseline — which one is preferable depends on whether average accuracy or
worst-case danger matters more for the deployment context.

In [ ]:
def stage_matched_predict(test_row, train_df, feature_cols, margin=20):
    stage = test_row['time_in_cycles']
    subset = train_df[
        (train_df['time_in_cycles'] >= stage - margin) &
        (train_df['time_in_cycles'] <= stage + margin)
    ]
    if len(subset) < 50:
        subset = train_df
    model = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, min_samples_leaf=5)
    model.fit(subset[feature_cols], subset['RUL'])
    return model.predict(test_row[feature_cols].values.reshape(1, -1))[0]

y_pred_stage = np.array([
    stage_matched_predict(test_last.iloc[i], train_split, feature_cols)
    for i in range(len(test_last))
])

_ = evaluate_full(y_true, y_pred_stage, label='Stage-matched (uncapped)')
y_pred_stage_capped = np.minimum(y_pred_stage, RUL_CAP)
_ = evaluate_full(y_true_capped, y_pred_stage_capped, label='Stage-matched (capped)')

## 10. Binary Classification: "Does This Engine Need Maintenance Soon?"

Reframing the problem from "predict the exact number" to "flag engines that need
attention" — a more actionable and, as shown below, more honestly evaluable output.

In [ ]:
MAINTENANCE_THRESHOLD = 20  # cycles

y_true_class = (y_true <= MAINTENANCE_THRESHOLD).astype(int)

train_split_c = train_split.copy()
train_split_c['label'] = (train_split_c['RUL'] <= MAINTENANCE_THRESHOLD).astype(int)

clf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42,
                              min_samples_leaf=5, class_weight='balanced')
clf.fit(train_split_c[feature_cols], train_split_c['label'])

y_pred_class = clf.predict(test_last[feature_cols])

print("Positive (needs maintenance) engines in test set:", y_true_class.sum(), "/", len(y_true_class))
print("Accuracy:", round(accuracy_score(y_true_class, y_pred_class), 3))
print("Precision:", round(precision_score(y_true_class, y_pred_class), 3))
print("Recall:", round(recall_score(y_true_class, y_pred_class), 3))
print("F1:", round(f1_score(y_true_class, y_pred_class), 3))

## 11. Expected Value: Translating Accuracy into Business Value

Using a cost-benefit framework (Data Science for Business): TP = correctly flagged
(benefit), FN = missed failure (high cost — unplanned downtime), FP = false alarm
(moderate cost — wasted maintenance visit), TN = correctly left alone (no cost).
Dollar values below are illustrative placeholders; in a real deployment they'd come
from domain/business input, but the *ranking* of strategies is the important result.

In [ ]:
def confusion_counts(y_true, y_pred):
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    return tp, fp, fn, tn

def expected_value(y_true, y_pred, tp_val=50000, fp_cost=-5000, fn_cost=-80000, tn_val=0, label=''):
    tp, fp, fn, tn = confusion_counts(y_true, y_pred)
    total = tp * tp_val + fp * fp_cost + fn * fn_cost + tn * tn_val
    print(f"[{label}] TP={tp}  FP={fp}  FN={fn}  TN={tn}  ->  Expected value = ${total:,}")
    return total

ev_model = expected_value(y_true_class, y_pred_class, label='Model')

## 12. The Critical Comparison: Model vs Naive Strategies

A dollar figure alone means little without a baseline. We compare against never
flagging anyone (pure reactive maintenance) and always flagging everyone (pure
preventive maintenance). The model's value proposition rests on beating both.

In [ ]:
y_pred_never = np.zeros_like(y_true_class)
ev_never = expected_value(y_true_class, y_pred_never, label='Never flag (reactive)')

y_pred_always = np.ones_like(y_true_class)
ev_always = expected_value(y_true_class, y_pred_always, label='Always flag (preventive)')

print()
print(f"Model's gain over reactive:   ${ev_model - ev_never:,}")
print(f"Model's gain over preventive: ${ev_model - ev_always:,}")

strategies = ['Reactive\n(never flag)', 'Preventive\n(always flag)', 'Our Model']
values = [ev_never, ev_always, ev_model]
colors = ['#c0392b', '#7f8c8d', '#2980b9']

plt.figure(figsize=(7, 4.5))
bars = plt.bar(strategies, values, color=colors)
plt.axhline(0, color='black', linewidth=0.8)
plt.ylabel('Expected Value ($)')
plt.title('Expected Value: Model vs Naive Strategies')
for bar, v in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, v, f"${v:,.0f}",
              ha='center', va='bottom' if v >= 0 else 'top', fontsize=9)
plt.tight_layout()
plt.show()

## 13. Early-Warning Test: Does the Model Actually Work Early in an Engine's Life?

Everything above evaluates each engine at whatever point NASA's official test set
happened to truncate it — which is *not* the same question as "if we only had early
data, would this still work?" This section answers that question directly, by forcing
a fixed cutoff (as a fraction of the average training engine's lifespan) and checking
whether the classifier still catches at-risk engines using only data up to that point.

In [ ]:
def evaluate_at_cutoff(cutoff_fraction, avg_life, test_df, y_test_official, clf, feature_cols, threshold=20):
    cutoff = int(avg_life * cutoff_fraction)
    t_early = test_df[test_df['time_in_cycles'] <= cutoff].sort_values('time_in_cycles').groupby('unit_number').last().reset_index()
    if len(t_early) == 0:
        return None

    t_full_len = test_df.groupby('unit_number')['time_in_cycles'].max().reset_index()
    t_full_len.columns = ['unit_number', 'observed_length']
    t_early = t_early.merge(t_full_len, on='unit_number')
    t_early = t_early.merge(
        y_test_official.reset_index().rename(columns={'index': 'unit_number_idx'}),
        left_index=True, right_on='unit_number_idx'
    )
    t_early['true_total_life'] = t_early['observed_length'] + t_early['RUL']
    t_early['true_RUL_at_cutoff'] = t_early['true_total_life'] - cutoff

    y_true_c = (t_early['true_RUL_at_cutoff'] <= threshold).astype(int)
    y_pred_c = clf.predict(t_early[feature_cols])

    n_pos = y_true_c.sum()
    acc = accuracy_score(y_true_c, y_pred_c)
    rec = recall_score(y_true_c, y_pred_c, zero_division=0)
    prec = precision_score(y_true_c, y_pred_c, zero_division=0)
    fp_rate = ((y_true_c == 0) & (y_pred_c == 1)).sum() / max((y_true_c == 0).sum(), 1)

    return {'cutoff_frac': cutoff_fraction, 'cutoff_cycles': cutoff, 'n_engines': len(t_early),
            'n_true_positive': int(n_pos), 'accuracy': round(acc, 3), 'recall': round(rec, 3),
            'precision': round(prec, 3), 'false_alarm_rate': round(fp_rate, 3)}

avg_life = train_split.groupby('unit_number')['time_in_cycles'].max().mean()
print(f"Average training engine lifespan: {avg_life:.1f} cycles\n")

results = []
for frac in [0.2, 0.4, 0.6, 0.8, 1.0]:
    r = evaluate_at_cutoff(frac, avg_life, test, y_test_official, clf, feature_cols)
    if r:
        results.append(r)

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

## 14. Honest Conclusion

**Finding:** recall never exceeds ~0.21-0.28 across any fixed early/mid cutoff — even
using the *full average lifespan* (207 cycles) of data. This stands in sharp contrast
to the recall (0.938) measured on NASA's official test set in Section 10, strongly
suggesting that the official test truncation points skew toward later (easier, more
obviously degraded) stages of engine life — inflating the apparent recall relative to
genuine early/mid-life performance.

We tested five interventions to close this gap (probability threshold sweeping,
regression-based flagging instead of direct classification, aggressive class
weighting, raw sensors only, and looser RUL thresholds from 20 to 50 cycles) — **none**
moved recall meaningfully above ~0.20-0.28. This points to a structural limitation of
the current feature set (raw + rolling-mean sensor statistics with a Random Forest),
not a tunable hyperparameter problem.

**What this model is honestly good for:**
- A low-false-alarm secondary signal (precision stays ~0.94-1.0 throughout)
- A reliable business-value case *once an engine is well into its degraded phase*
  (Section 10-12's expected-value comparison holds at that stage)

**What it is not good for:**
- A primary early-warning system — it misses the majority of at-risk engines when
  only early-to-mid-life data is available.

Closing this gap would likely require sequence-aware models (e.g. LSTM over raw
time windows) or richer physics-informed features — both out of scope for this
notebook's goal of a fast, interpretable, honestly-evaluated baseline. This finding
should be carried forward transparently into any dashboard built on this model: it
is reliable at flagging problems that are already fairly advanced, not at catching
them early.